# mini hw3 - ensembles

In [54]:
from sklearn import model_selection,linear_model,metrics
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn import decomposition,ensemble
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from scipy.optimize import fmin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from functools import partial
import xgboost

from sklearn.preprocessing import *
from sklearn import preprocessing

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import Pipeline
from mlxtend.regressor import StackingCVRegressor
from sklearn.svm import SVR
from sklearn.impute import SimpleImputer
from imblearn import FunctionSampler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import numpy as np 
import pandas as pd 
import seaborn as sns

import glob
import time
import os

import warnings
warnings.filterwarnings("ignore")

## Подготовка данных

Загрузите и предобработайте данные (по своему усмотрению) из hw1

In [74]:
train = pd.read_csv('train_features_with_answers-Copy1.csv')
testing = pd.read_csv('X_test-Copy1.csv')

train['sex'] = train['sex'].map(lambda x: np.nan if ((x != 'M') & (x != 'F')) else x)
train['age'] = train['age'].map(lambda x: np.nan if ((14>x) | (x>100)) else x)
train['sex'] = train['sex'].map(lambda x: 1 if x == 'M' else 0 if x == 'F' else x)
train['address'] = train['address'].map(lambda x: 1 if x == 'U' else 0 if x == 'R' else x)

train_x = train.drop('G3', axis=1)
train_y = train['G3']

train_space = train_x.select_dtypes(exclude=['object'])
train_space = train_space[train_space['sex'].notna()]
train_space = train_space[train_space['age'].notna()]
train_space = train_space[train_space['address'].notna()]

train_sex_x = train_space.drop('sex', axis=1).drop('age', axis=1).drop('address', axis=1)
train_sex_y = train_space['sex']

new_model = KNeighborsClassifier(n_neighbors = 5)

x_training_data, x_test_data, y_training_data, y_test_data = train_test_split(train_sex_x, train_sex_y, test_size = 0.3)
new_model.fit(x_training_data, y_training_data)
new_predictions = new_model.predict(x_test_data)
accuracy_score(y_test_data, new_predictions)

test_sex = train_x[train_x['sex'].isnull()].select_dtypes(exclude=['object']).drop('sex', axis = 1).drop('address', axis = 1).drop('age', axis = 1)

train_x.loc[test_sex.index, ['sex']] = new_model.predict(test_sex)

train_sex_x = train_space.drop('sex', axis=1).drop('age', axis=1).drop('address', axis=1)
train_sex_y = train_space['address']

new_model = KNeighborsClassifier(n_neighbors = 5)

x_training_data, x_test_data, y_training_data, y_test_data = train_test_split(train_sex_x, train_sex_y, test_size = 0.3)
new_model.fit(x_training_data, y_training_data)
new_predictions = new_model.predict(x_test_data)
accuracy_score(y_test_data, new_predictions)

test_sex = train_x[train_x['address'].isnull()].select_dtypes(exclude=['object']).drop('sex', axis = 1).drop('address', axis = 1).drop('age', axis = 1)

train_x.loc[test_sex.index, ['address']] = new_model.predict(test_sex)

train_x1 = train_x.copy()
for c in train_x.columns:
    if train_x1[c].dtype == np.object:
        train_x1 = pd.get_dummies(train_x1, columns=[c])
train_x = train_x1.copy()

train_x = train_x.drop('failures', axis = 1)
train_x = train_x[(train_x['Dalc'] <4) & (train_x['famrel'] > 2 ) & (train_x["absences"] != 24) 
                    & (train_x["freetime"] > 1) & (train_x["studytime"] < 4) & (train_x["traveltime"] < 4)
                   & (train_x["age"] < 21)]

train_y = train_y.iloc[train_x.index]

corr_matrix = train_x.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.5)]
train_x.drop(to_drop, axis=1, inplace=True)

testing['sex'] = testing['sex'].map(lambda x: 1 if x == 'M' else 0 if x == 'F' else x)
testing['address'] = testing['address'].map(lambda x: 1 if x == 'U' else 0 if x == 'R' else x)
testing = testing.drop('failures', axis = 1)

test2 = testing.copy()

for c in testing.columns:
    if test2[c].dtype == np.object:
        test2 = pd.get_dummies(test2, columns=[c])
test2.drop(to_drop, axis=1, inplace=True)

testing = test2
    
training = pd.concat([train_x, train_y], axis=1)

train, test = train_test_split(training, test_size = 0.3)

encoder = LabelEncoder()
train_label = train.copy()
test_label = test.copy()

for i in range(10):
    train_label.iloc[:,i] = encoder.fit_transform(train.iloc[:,i])
    test_label.iloc[:,i] = encoder.transform(test.iloc[:,i])
    
X = train_label.drop('G3',axis=1)
y = train_label.G3

## Обоснуйте выбор слабых (базовых) алгоритмов

Бустинги вроде как раз и созданы для объединения слабых алгоритмов в сильный, взяла все что нашлись и скачались(F to xgb)

In [134]:
li_reg = LinearRegression()
gb_reg = GradientBoostingRegressor(n_estimators=1000, max_depth=3)

cgb_reg=CatBoostRegressor(n_estimators=5000, learning_rate=0.01)
lgb_reg = LGBMRegressor(learning_rate = 0.003,
                                n_estimators = 16000,
                                num_leaves = 20,
                                max_depth = 20,
                                n_jobs = -1,
                                random_state = 42)

In [135]:
li_reg.fit(X, y)
gb_reg.fit(X , y)
cgb_reg.fit(X, y)
lgb_reg.fit(X, y)
pass

0:	learn: 3.1096648	total: 993us	remaining: 4.96s
1:	learn: 3.1035883	total: 2.23ms	remaining: 5.58s
2:	learn: 3.0971396	total: 3.51ms	remaining: 5.85s
3:	learn: 3.0909230	total: 4.86ms	remaining: 6.07s
4:	learn: 3.0856119	total: 6.13ms	remaining: 6.13s
5:	learn: 3.0791451	total: 7.26ms	remaining: 6.04s
6:	learn: 3.0733382	total: 8.41ms	remaining: 6s
7:	learn: 3.0677251	total: 9.6ms	remaining: 5.99s
8:	learn: 3.0628305	total: 10.9ms	remaining: 6.05s
9:	learn: 3.0564936	total: 12.2ms	remaining: 6.1s
10:	learn: 3.0502222	total: 13.8ms	remaining: 6.25s
11:	learn: 3.0433057	total: 15ms	remaining: 6.22s
12:	learn: 3.0369891	total: 16.2ms	remaining: 6.23s
13:	learn: 3.0312144	total: 17.6ms	remaining: 6.28s
14:	learn: 3.0249225	total: 18.6ms	remaining: 6.2s
15:	learn: 3.0201034	total: 20.3ms	remaining: 6.32s
16:	learn: 3.0147754	total: 21.3ms	remaining: 6.23s
17:	learn: 3.0084255	total: 22.7ms	remaining: 6.3s
18:	learn: 3.0030193	total: 24.2ms	remaining: 6.34s
19:	learn: 2.9980450	total: 25.1

213:	learn: 2.2682745	total: 314ms	remaining: 7.02s
214:	learn: 2.2656445	total: 315ms	remaining: 7.02s
215:	learn: 2.2609587	total: 317ms	remaining: 7.02s
216:	learn: 2.2585881	total: 319ms	remaining: 7.02s
217:	learn: 2.2551028	total: 320ms	remaining: 7.01s
218:	learn: 2.2522157	total: 321ms	remaining: 7.01s
219:	learn: 2.2495711	total: 323ms	remaining: 7.01s
220:	learn: 2.2474352	total: 324ms	remaining: 7s
221:	learn: 2.2442092	total: 325ms	remaining: 7s
222:	learn: 2.2412285	total: 327ms	remaining: 7s
223:	learn: 2.2393554	total: 328ms	remaining: 6.99s
224:	learn: 2.2361858	total: 330ms	remaining: 6.99s
225:	learn: 2.2325109	total: 331ms	remaining: 6.99s
226:	learn: 2.2290735	total: 332ms	remaining: 6.99s
227:	learn: 2.2252236	total: 334ms	remaining: 6.99s
228:	learn: 2.2214587	total: 336ms	remaining: 6.99s
229:	learn: 2.2185825	total: 337ms	remaining: 6.99s
230:	learn: 2.2161502	total: 338ms	remaining: 6.99s
231:	learn: 2.2140571	total: 340ms	remaining: 6.99s
232:	learn: 2.2112536

415:	learn: 1.8204795	total: 628ms	remaining: 6.92s
416:	learn: 1.8188771	total: 629ms	remaining: 6.91s
417:	learn: 1.8171351	total: 631ms	remaining: 6.92s
418:	learn: 1.8149171	total: 633ms	remaining: 6.92s
419:	learn: 1.8136210	total: 634ms	remaining: 6.92s
420:	learn: 1.8114141	total: 636ms	remaining: 6.91s
421:	learn: 1.8092979	total: 637ms	remaining: 6.91s
422:	learn: 1.8074838	total: 639ms	remaining: 6.91s
423:	learn: 1.8055672	total: 640ms	remaining: 6.91s
424:	learn: 1.8031165	total: 642ms	remaining: 6.91s
425:	learn: 1.8012830	total: 643ms	remaining: 6.91s
426:	learn: 1.7990746	total: 645ms	remaining: 6.9s
427:	learn: 1.7979318	total: 646ms	remaining: 6.91s
428:	learn: 1.7962715	total: 648ms	remaining: 6.9s
429:	learn: 1.7943598	total: 649ms	remaining: 6.9s
430:	learn: 1.7928369	total: 651ms	remaining: 6.9s
431:	learn: 1.7904642	total: 653ms	remaining: 6.9s
432:	learn: 1.7890833	total: 654ms	remaining: 6.9s
433:	learn: 1.7879353	total: 656ms	remaining: 6.9s
434:	learn: 1.78645

621:	learn: 1.4844656	total: 958ms	remaining: 6.74s
622:	learn: 1.4827567	total: 960ms	remaining: 6.74s
623:	learn: 1.4819061	total: 962ms	remaining: 6.74s
624:	learn: 1.4799222	total: 963ms	remaining: 6.74s
625:	learn: 1.4782479	total: 965ms	remaining: 6.74s
626:	learn: 1.4767356	total: 966ms	remaining: 6.74s
627:	learn: 1.4753212	total: 968ms	remaining: 6.74s
628:	learn: 1.4737366	total: 970ms	remaining: 6.74s
629:	learn: 1.4726408	total: 1s	remaining: 6.96s
630:	learn: 1.4722146	total: 1s	remaining: 6.95s
631:	learn: 1.4703716	total: 1s	remaining: 6.95s
632:	learn: 1.4685804	total: 1.01s	remaining: 6.95s
633:	learn: 1.4674805	total: 1.01s	remaining: 6.95s
634:	learn: 1.4658370	total: 1.01s	remaining: 6.94s
635:	learn: 1.4644872	total: 1.01s	remaining: 6.94s
636:	learn: 1.4633135	total: 1.01s	remaining: 6.93s
637:	learn: 1.4621208	total: 1.01s	remaining: 6.93s
638:	learn: 1.4609081	total: 1.01s	remaining: 6.93s
639:	learn: 1.4595835	total: 1.02s	remaining: 6.93s
640:	learn: 1.4583522

803:	learn: 1.2444394	total: 1.27s	remaining: 6.63s
804:	learn: 1.2440315	total: 1.27s	remaining: 6.63s
805:	learn: 1.2433534	total: 1.27s	remaining: 6.63s
806:	learn: 1.2424158	total: 1.27s	remaining: 6.63s
807:	learn: 1.2412500	total: 1.28s	remaining: 6.63s
808:	learn: 1.2406417	total: 1.28s	remaining: 6.63s
809:	learn: 1.2394313	total: 1.28s	remaining: 6.62s
810:	learn: 1.2385088	total: 1.28s	remaining: 6.62s
811:	learn: 1.2371552	total: 1.28s	remaining: 6.62s
812:	learn: 1.2365614	total: 1.28s	remaining: 6.62s
813:	learn: 1.2350067	total: 1.29s	remaining: 6.62s
814:	learn: 1.2333461	total: 1.29s	remaining: 6.62s
815:	learn: 1.2323184	total: 1.29s	remaining: 6.62s
816:	learn: 1.2308948	total: 1.29s	remaining: 6.61s
817:	learn: 1.2302506	total: 1.29s	remaining: 6.61s
818:	learn: 1.2288419	total: 1.29s	remaining: 6.61s
819:	learn: 1.2278748	total: 1.3s	remaining: 6.61s
820:	learn: 1.2268613	total: 1.3s	remaining: 6.6s
821:	learn: 1.2258472	total: 1.3s	remaining: 6.6s
822:	learn: 1.224

...

4160:	learn: 0.1081278	total: 6.58s	remaining: 1.33s
4161:	learn: 0.1080950	total: 6.58s	remaining: 1.32s
4162:	learn: 0.1080561	total: 6.58s	remaining: 1.32s
4163:	learn: 0.1079728	total: 6.59s	remaining: 1.32s
4164:	learn: 0.1079561	total: 6.59s	remaining: 1.32s
4165:	learn: 0.1078963	total: 6.59s	remaining: 1.32s
4166:	learn: 0.1078407	total: 6.59s	remaining: 1.32s
4167:	learn: 0.1077649	total: 6.59s	remaining: 1.32s
4168:	learn: 0.1076246	total: 6.6s	remaining: 1.31s
4169:	learn: 0.1075338	total: 6.6s	remaining: 1.31s
4170:	learn: 0.1074213	total: 6.6s	remaining: 1.31s
4171:	learn: 0.1073273	total: 6.6s	remaining: 1.31s
4172:	learn: 0.1072335	total: 6.6s	remaining: 1.31s
4173:	learn: 0.1071513	total: 6.61s	remaining: 1.31s
4174:	learn: 0.1070732	total: 6.61s	remaining: 1.3s
4175:	learn: 0.1069947	total: 6.61s	remaining: 1.3s
4176:	learn: 0.1069570	total: 6.61s	remaining: 1.3s
4177:	learn: 0.1068678	total: 6.61s	remaining: 1.3s
4178:	learn: 0.1068378	total: 6.61s	remaining: 1.3s
417

4341:	learn: 0.0951669	total: 6.89s	remaining: 1.04s
4342:	learn: 0.0951101	total: 6.89s	remaining: 1.04s
4343:	learn: 0.0950471	total: 6.89s	remaining: 1.04s
4344:	learn: 0.0950170	total: 6.89s	remaining: 1.04s
4345:	learn: 0.0949421	total: 6.9s	remaining: 1.04s
4346:	learn: 0.0948795	total: 6.9s	remaining: 1.04s
4347:	learn: 0.0947801	total: 6.9s	remaining: 1.03s
4348:	learn: 0.0946762	total: 6.9s	remaining: 1.03s
4349:	learn: 0.0945552	total: 6.9s	remaining: 1.03s
4350:	learn: 0.0945354	total: 6.91s	remaining: 1.03s
4351:	learn: 0.0944949	total: 6.91s	remaining: 1.03s
4352:	learn: 0.0944478	total: 6.91s	remaining: 1.03s
4353:	learn: 0.0943640	total: 6.91s	remaining: 1.02s
4354:	learn: 0.0942929	total: 6.91s	remaining: 1.02s
4355:	learn: 0.0942535	total: 6.91s	remaining: 1.02s
4356:	learn: 0.0941898	total: 6.92s	remaining: 1.02s
4357:	learn: 0.0941137	total: 6.92s	remaining: 1.02s
4358:	learn: 0.0940677	total: 6.92s	remaining: 1.02s
4359:	learn: 0.0940045	total: 6.92s	remaining: 1.01

4589:	learn: 0.0806072	total: 7.37s	remaining: 659ms
4590:	learn: 0.0805627	total: 7.38s	remaining: 657ms
4591:	learn: 0.0804860	total: 7.38s	remaining: 655ms
4592:	learn: 0.0804209	total: 7.38s	remaining: 654ms
4593:	learn: 0.0803515	total: 7.38s	remaining: 652ms
4594:	learn: 0.0802805	total: 7.38s	remaining: 651ms
4595:	learn: 0.0802126	total: 7.38s	remaining: 649ms
4596:	learn: 0.0801536	total: 7.38s	remaining: 647ms
4597:	learn: 0.0800758	total: 7.39s	remaining: 646ms
4598:	learn: 0.0800388	total: 7.39s	remaining: 644ms
4599:	learn: 0.0799946	total: 7.39s	remaining: 643ms
4600:	learn: 0.0799280	total: 7.39s	remaining: 641ms
4601:	learn: 0.0798452	total: 7.39s	remaining: 639ms
4602:	learn: 0.0797620	total: 7.39s	remaining: 638ms
4603:	learn: 0.0796606	total: 7.39s	remaining: 636ms
4604:	learn: 0.0796326	total: 7.4s	remaining: 634ms
4605:	learn: 0.0796055	total: 7.4s	remaining: 633ms
4606:	learn: 0.0795517	total: 7.4s	remaining: 631ms
4607:	learn: 0.0794827	total: 7.4s	remaining: 630

4760:	learn: 0.0710049	total: 7.67s	remaining: 385ms
4761:	learn: 0.0709146	total: 7.68s	remaining: 384ms
4762:	learn: 0.0708739	total: 7.68s	remaining: 382ms
4763:	learn: 0.0708243	total: 7.68s	remaining: 380ms
4764:	learn: 0.0708090	total: 7.68s	remaining: 379ms
4765:	learn: 0.0707321	total: 7.68s	remaining: 377ms
4766:	learn: 0.0707128	total: 7.68s	remaining: 376ms
4767:	learn: 0.0706611	total: 7.69s	remaining: 374ms
4768:	learn: 0.0706309	total: 7.69s	remaining: 372ms
4769:	learn: 0.0705927	total: 7.69s	remaining: 371ms
4770:	learn: 0.0705514	total: 7.69s	remaining: 369ms
4771:	learn: 0.0704365	total: 7.69s	remaining: 368ms
4772:	learn: 0.0704115	total: 7.7s	remaining: 366ms
4773:	learn: 0.0703460	total: 7.7s	remaining: 364ms
4774:	learn: 0.0702844	total: 7.7s	remaining: 363ms
4775:	learn: 0.0702350	total: 7.7s	remaining: 361ms
4776:	learn: 0.0701848	total: 7.7s	remaining: 360ms
4777:	learn: 0.0701431	total: 7.71s	remaining: 358ms
4778:	learn: 0.0700826	total: 7.71s	remaining: 356m

4938:	learn: 0.0622182	total: 7.97s	remaining: 98.5ms
4939:	learn: 0.0621553	total: 7.98s	remaining: 96.9ms
4940:	learn: 0.0620820	total: 7.98s	remaining: 95.3ms
4941:	learn: 0.0620092	total: 7.98s	remaining: 93.7ms
4942:	learn: 0.0619846	total: 7.98s	remaining: 92ms
4943:	learn: 0.0619409	total: 7.98s	remaining: 90.4ms
4944:	learn: 0.0618778	total: 7.98s	remaining: 88.8ms
4945:	learn: 0.0618185	total: 7.99s	remaining: 87.2ms
4946:	learn: 0.0617711	total: 7.99s	remaining: 85.6ms
4947:	learn: 0.0617283	total: 7.99s	remaining: 84ms
4948:	learn: 0.0616866	total: 7.99s	remaining: 82.4ms
4949:	learn: 0.0616000	total: 7.99s	remaining: 80.7ms
4950:	learn: 0.0615768	total: 8s	remaining: 79.1ms
4951:	learn: 0.0615398	total: 8s	remaining: 77.6ms
4952:	learn: 0.0615057	total: 8.01s	remaining: 76ms
4953:	learn: 0.0614539	total: 8.01s	remaining: 74.3ms
4954:	learn: 0.0614120	total: 8.01s	remaining: 72.7ms
4955:	learn: 0.0613672	total: 8.01s	remaining: 71.1ms
4956:	learn: 0.0613134	total: 8.01s	rema

In [136]:
li_pred = li_reg.predict(test_label.drop('G3',axis=1))
gb_pred = gb_reg.predict(test_label.drop('G3',axis=1))
cgb_pred = cgb_reg.predict(test_label.drop('G3',axis=1))
lgb_pred = lgb_reg.predict(test_label.drop('G3',axis=1))

In [137]:
print("Alg --- MSE --- R2 --- Accuracy")

for s in [li_pred, gb_pred, cgb_pred, lgb_pred]:
    
    print([k for k in globals() if s is globals()[k]][0][:-5], round(mean_squared_error(np.round(s, 0), test_label.G3), 2), round(r2_score(test_label.G3, np.round(s, 0)), 2), round(accuracy_score(test_label.G3, np.round(s, 0)), 2), sep = "    ")

Alg --- MSE --- R2 --- Accuracy
li    6.84    0.22    0.13
gb    7.8    0.11    0.13
cgb    7.38    0.16    0.2
lgb    7.9    0.1    0.17


In [138]:
def mse(s):
    return mean_squared_error(s, test_label.G3)
def r2(s):
    return r2_score(test_label.G3, s)
def acc(s):
    return accuracy_score(test_label.G3, np.round(s, 0))

## Постройте решение на основе подхода Blending

Правила:
- Нужно использовать вероятности
- Предложите что-то лучше, чем брать среднее от предсказаний моделей (оценивать уверенность алгоритмов, точности и т.д.)
- Заставьте базовые алгоритмы быть некорелированными
- Добавьте рандома (например, стройте ваши алгоритмы на разных выборках, по разному предобрабатывайте данные или применяйте для разных признаков соответствующие алгоритмы ... )
- Проявите смекалку

In [139]:
blend_a=(li_pred+gb_pred+cgb_pred+lgb_pred)/4 #mean
blend_b=(li_pred*gb_pred*cgb_pred*lgb_pred)**(1/4) ##geom mean

blend_c=(gb_pred+cgb_pred+lgb_pred)/3 #only boost
blend_d=(gb_pred*cgb_pred*lgb_pred)**(1/3)

blend_e=(li_pred*acc(li_pred)+gb_pred*acc(gb_pred)+cgb_pred*acc(cgb_pred)+lgb_pred*acc(lgb_pred))/(acc(li_pred)+acc(gb_pred)+acc(cgb_pred)+acc(lgb_pred))
blend_f=(li_pred*r2(li_pred)+gb_pred*r2(gb_pred)+cgb_pred*r2(cgb_pred)+lgb_pred*r2(lgb_pred))/(r2(li_pred)+r2(gb_pred)+r2(cgb_pred)+r2(lgb_pred))
blend_g=(li_pred/mse(li_pred)+gb_pred/mse(gb_pred)+cgb_pred/mse(cgb_pred)+lgb_pred/mse(lgb_pred))/(1/mse(li_pred)+1/mse(gb_pred)+1/mse(cgb_pred)+1/mse(lgb_pred))

In [157]:
print("Alg ------ MSE --- R2 --- Accuracy")

for s in [globals()["blend_"+s] for s in ["a", "b", "c", "d", "e", "f", "g"]]:
    
    print([k for k in globals() if s is globals()[k]][1], round(mean_squared_error(np.round(s, 0), test_label.G3), 2), round(r2_score(test_label.G3, np.round(s, 0)), 2), round(accuracy_score(test_label.G3, np.round(s, 0)), 2), sep = "    ")

Alg ------ MSE --- R2 --- Accuracy
blend_a    6.79    0.23    0.24
blend_b    6.77    0.23    0.23
blend_c    7.09    0.2    0.16
blend_d    7.0    0.21    0.17
blend_e    6.7    0.24    0.24
blend_f    6.72    0.24    0.26
blend_g    6.79    0.23    0.24


#a few random?

In [171]:
for s in [li_reg, gb_reg, cgb_reg, lgb_reg]:
    X_1, _, y_1, _ = train_test_split(X, y, test_size = 0.3)
    s.fit(X_1, y_1)
    globals()[[k for k in globals() if s is globals()[k]][0][:-4] + '_pred'] = s.predict(test_label.drop('G3',axis=1))

0:	learn: 3.1769764	total: 1.45ms	remaining: 7.23s
1:	learn: 3.1720825	total: 3.29ms	remaining: 8.21s
2:	learn: 3.1653915	total: 4.75ms	remaining: 7.92s
3:	learn: 3.1581895	total: 6.28ms	remaining: 7.85s
4:	learn: 3.1535621	total: 7.92ms	remaining: 7.91s
5:	learn: 3.1466708	total: 9.33ms	remaining: 7.77s
6:	learn: 3.1401769	total: 10.8ms	remaining: 7.7s
7:	learn: 3.1338857	total: 12.3ms	remaining: 7.7s
8:	learn: 3.1292237	total: 13.7ms	remaining: 7.57s
9:	learn: 3.1222806	total: 15ms	remaining: 7.49s
10:	learn: 3.1158236	total: 16.5ms	remaining: 7.47s
11:	learn: 3.1096050	total: 18ms	remaining: 7.48s
12:	learn: 3.1025725	total: 19.5ms	remaining: 7.48s
13:	learn: 3.0959391	total: 20.9ms	remaining: 7.46s
14:	learn: 3.0887863	total: 22.4ms	remaining: 7.46s
15:	learn: 3.0803535	total: 24.1ms	remaining: 7.51s
16:	learn: 3.0756243	total: 25.7ms	remaining: 7.53s
17:	learn: 3.0696045	total: 27.2ms	remaining: 7.51s
18:	learn: 3.0630193	total: 28.6ms	remaining: 7.5s
19:	learn: 3.0574240	total: 3

209:	learn: 2.2049279	total: 314ms	remaining: 7.16s
210:	learn: 2.2002868	total: 316ms	remaining: 7.17s
211:	learn: 2.1969647	total: 318ms	remaining: 7.17s
212:	learn: 2.1931435	total: 319ms	remaining: 7.17s
213:	learn: 2.1886817	total: 320ms	remaining: 7.17s
214:	learn: 2.1855590	total: 322ms	remaining: 7.16s
215:	learn: 2.1825302	total: 323ms	remaining: 7.16s
216:	learn: 2.1793703	total: 325ms	remaining: 7.16s
217:	learn: 2.1771029	total: 326ms	remaining: 7.16s
218:	learn: 2.1735220	total: 328ms	remaining: 7.16s
219:	learn: 2.1713900	total: 330ms	remaining: 7.16s
220:	learn: 2.1683111	total: 331ms	remaining: 7.16s
221:	learn: 2.1652793	total: 333ms	remaining: 7.17s
222:	learn: 2.1623217	total: 335ms	remaining: 7.18s
223:	learn: 2.1584143	total: 337ms	remaining: 7.18s
224:	learn: 2.1551030	total: 339ms	remaining: 7.18s
225:	learn: 2.1518447	total: 340ms	remaining: 7.19s
226:	learn: 2.1491516	total: 342ms	remaining: 7.19s
227:	learn: 2.1462496	total: 344ms	remaining: 7.19s
228:	learn: 

398:	learn: 1.7207614	total: 629ms	remaining: 7.25s
399:	learn: 1.7186440	total: 631ms	remaining: 7.25s
400:	learn: 1.7176899	total: 632ms	remaining: 7.25s
401:	learn: 1.7154367	total: 634ms	remaining: 7.25s
402:	learn: 1.7128845	total: 635ms	remaining: 7.25s
403:	learn: 1.7106378	total: 637ms	remaining: 7.25s
404:	learn: 1.7093298	total: 639ms	remaining: 7.25s
405:	learn: 1.7083777	total: 640ms	remaining: 7.25s
406:	learn: 1.7067934	total: 642ms	remaining: 7.25s
407:	learn: 1.7049694	total: 644ms	remaining: 7.25s
408:	learn: 1.7028086	total: 646ms	remaining: 7.25s
409:	learn: 1.7011615	total: 647ms	remaining: 7.25s
410:	learn: 1.6985158	total: 649ms	remaining: 7.24s
411:	learn: 1.6970642	total: 650ms	remaining: 7.24s
412:	learn: 1.6964317	total: 651ms	remaining: 7.23s
413:	learn: 1.6951380	total: 652ms	remaining: 7.22s
414:	learn: 1.6933143	total: 653ms	remaining: 7.22s
415:	learn: 1.6917764	total: 655ms	remaining: 7.22s
416:	learn: 1.6894574	total: 656ms	remaining: 7.21s
417:	learn: 

606:	learn: 1.3457339	total: 936ms	remaining: 6.77s
607:	learn: 1.3438063	total: 937ms	remaining: 6.77s
608:	learn: 1.3427852	total: 939ms	remaining: 6.77s
609:	learn: 1.3408345	total: 940ms	remaining: 6.77s
610:	learn: 1.3400221	total: 942ms	remaining: 6.76s
611:	learn: 1.3381995	total: 943ms	remaining: 6.76s
612:	learn: 1.3364217	total: 945ms	remaining: 6.76s
613:	learn: 1.3349155	total: 946ms	remaining: 6.76s
614:	learn: 1.3331841	total: 948ms	remaining: 6.76s
615:	learn: 1.3319285	total: 950ms	remaining: 6.76s
616:	learn: 1.3303720	total: 952ms	remaining: 6.76s
617:	learn: 1.3283061	total: 953ms	remaining: 6.76s
618:	learn: 1.3261837	total: 955ms	remaining: 6.76s
619:	learn: 1.3243576	total: 956ms	remaining: 6.75s
620:	learn: 1.3229613	total: 958ms	remaining: 6.75s
621:	learn: 1.3218561	total: 959ms	remaining: 6.75s
622:	learn: 1.3211132	total: 961ms	remaining: 6.75s
623:	learn: 1.3194676	total: 962ms	remaining: 6.75s
624:	learn: 1.3173768	total: 964ms	remaining: 6.74s
625:	learn: 

812:	learn: 1.0818064	total: 1.25s	remaining: 6.42s
813:	learn: 1.0807733	total: 1.25s	remaining: 6.42s
814:	learn: 1.0796729	total: 1.25s	remaining: 6.42s
815:	learn: 1.0786494	total: 1.25s	remaining: 6.42s
816:	learn: 1.0779637	total: 1.25s	remaining: 6.42s
817:	learn: 1.0771247	total: 1.25s	remaining: 6.42s
818:	learn: 1.0759230	total: 1.26s	remaining: 6.42s
819:	learn: 1.0746804	total: 1.26s	remaining: 6.42s
820:	learn: 1.0732765	total: 1.26s	remaining: 6.42s
821:	learn: 1.0722373	total: 1.26s	remaining: 6.42s
822:	learn: 1.0705130	total: 1.26s	remaining: 6.42s
823:	learn: 1.0697413	total: 1.27s	remaining: 6.42s
824:	learn: 1.0685209	total: 1.27s	remaining: 6.42s
825:	learn: 1.0676521	total: 1.27s	remaining: 6.42s
826:	learn: 1.0669250	total: 1.27s	remaining: 6.41s
827:	learn: 1.0658215	total: 1.27s	remaining: 6.41s
828:	learn: 1.0650279	total: 1.27s	remaining: 6.41s
829:	learn: 1.0633813	total: 1.27s	remaining: 6.41s
830:	learn: 1.0624589	total: 1.28s	remaining: 6.41s
831:	learn: 

...

4028:	learn: 0.0765840	total: 6.24s	remaining: 1.5s
4029:	learn: 0.0765017	total: 6.24s	remaining: 1.5s
4030:	learn: 0.0764444	total: 6.24s	remaining: 1.5s
4031:	learn: 0.0764060	total: 6.24s	remaining: 1.5s
4032:	learn: 0.0763845	total: 6.24s	remaining: 1.5s
4033:	learn: 0.0763106	total: 6.24s	remaining: 1.5s
4034:	learn: 0.0762276	total: 6.25s	remaining: 1.49s
4035:	learn: 0.0761860	total: 6.25s	remaining: 1.49s
4036:	learn: 0.0760983	total: 6.25s	remaining: 1.49s
4037:	learn: 0.0760229	total: 6.25s	remaining: 1.49s
4038:	learn: 0.0759367	total: 6.25s	remaining: 1.49s
4039:	learn: 0.0758539	total: 6.25s	remaining: 1.49s
4040:	learn: 0.0757916	total: 6.25s	remaining: 1.48s
4041:	learn: 0.0756874	total: 6.26s	remaining: 1.48s
4042:	learn: 0.0755765	total: 6.26s	remaining: 1.48s
4043:	learn: 0.0754937	total: 6.26s	remaining: 1.48s
4044:	learn: 0.0754315	total: 6.26s	remaining: 1.48s
4045:	learn: 0.0753786	total: 6.26s	remaining: 1.48s
4046:	learn: 0.0753225	total: 6.26s	remaining: 1.48s

4230:	learn: 0.0635118	total: 6.55s	remaining: 1.19s
4231:	learn: 0.0634431	total: 6.55s	remaining: 1.19s
4232:	learn: 0.0633989	total: 6.55s	remaining: 1.19s
4233:	learn: 0.0633829	total: 6.55s	remaining: 1.19s
4234:	learn: 0.0633192	total: 6.55s	remaining: 1.18s
4235:	learn: 0.0632839	total: 6.56s	remaining: 1.18s
4236:	learn: 0.0632324	total: 6.56s	remaining: 1.18s
4237:	learn: 0.0631914	total: 6.56s	remaining: 1.18s
4238:	learn: 0.0631142	total: 6.56s	remaining: 1.18s
4239:	learn: 0.0630774	total: 6.56s	remaining: 1.18s
4240:	learn: 0.0629967	total: 6.56s	remaining: 1.17s
4241:	learn: 0.0629416	total: 6.57s	remaining: 1.17s
4242:	learn: 0.0629185	total: 6.57s	remaining: 1.17s
4243:	learn: 0.0628895	total: 6.57s	remaining: 1.17s
4244:	learn: 0.0628107	total: 6.57s	remaining: 1.17s
4245:	learn: 0.0627558	total: 6.57s	remaining: 1.17s
4246:	learn: 0.0627331	total: 6.57s	remaining: 1.17s
4247:	learn: 0.0626668	total: 6.57s	remaining: 1.16s
4248:	learn: 0.0625870	total: 6.58s	remaining:

4434:	learn: 0.0519783	total: 6.86s	remaining: 874ms
4435:	learn: 0.0519595	total: 6.87s	remaining: 873ms
4436:	learn: 0.0519268	total: 6.87s	remaining: 871ms
4437:	learn: 0.0518687	total: 6.87s	remaining: 870ms
4438:	learn: 0.0518183	total: 6.87s	remaining: 868ms
4439:	learn: 0.0517759	total: 6.87s	remaining: 867ms
4440:	learn: 0.0517492	total: 6.87s	remaining: 865ms
4441:	learn: 0.0517052	total: 6.87s	remaining: 864ms
4442:	learn: 0.0516655	total: 6.88s	remaining: 862ms
4443:	learn: 0.0516274	total: 6.88s	remaining: 860ms
4444:	learn: 0.0515744	total: 6.88s	remaining: 859ms
4445:	learn: 0.0515248	total: 6.88s	remaining: 857ms
4446:	learn: 0.0514944	total: 6.88s	remaining: 856ms
4447:	learn: 0.0514686	total: 6.88s	remaining: 854ms
4448:	learn: 0.0514080	total: 6.88s	remaining: 853ms
4449:	learn: 0.0513782	total: 6.89s	remaining: 851ms
4450:	learn: 0.0512989	total: 6.89s	remaining: 850ms
4451:	learn: 0.0512737	total: 6.89s	remaining: 848ms
4452:	learn: 0.0512074	total: 6.89s	remaining:

4643:	learn: 0.0427416	total: 7.18s	remaining: 550ms
4644:	learn: 0.0426883	total: 7.18s	remaining: 549ms
4645:	learn: 0.0426409	total: 7.18s	remaining: 547ms
4646:	learn: 0.0426138	total: 7.18s	remaining: 545ms
4647:	learn: 0.0425602	total: 7.18s	remaining: 544ms
4648:	learn: 0.0425350	total: 7.18s	remaining: 542ms
4649:	learn: 0.0424963	total: 7.18s	remaining: 541ms
4650:	learn: 0.0424560	total: 7.19s	remaining: 539ms
4651:	learn: 0.0424160	total: 7.19s	remaining: 538ms
4652:	learn: 0.0423643	total: 7.19s	remaining: 536ms
4653:	learn: 0.0423140	total: 7.19s	remaining: 535ms
4654:	learn: 0.0422863	total: 7.19s	remaining: 533ms
4655:	learn: 0.0422289	total: 7.2s	remaining: 532ms
4656:	learn: 0.0421720	total: 7.2s	remaining: 530ms
4657:	learn: 0.0421243	total: 7.2s	remaining: 529ms
4658:	learn: 0.0421027	total: 7.2s	remaining: 527ms
4659:	learn: 0.0420438	total: 7.2s	remaining: 525ms
4660:	learn: 0.0420180	total: 7.2s	remaining: 524ms
4661:	learn: 0.0419641	total: 7.2s	remaining: 522ms


4841:	learn: 0.0349581	total: 7.49s	remaining: 244ms
4842:	learn: 0.0349229	total: 7.49s	remaining: 243ms
4843:	learn: 0.0348803	total: 7.49s	remaining: 241ms
4844:	learn: 0.0348483	total: 7.49s	remaining: 240ms
4845:	learn: 0.0348221	total: 7.49s	remaining: 238ms
4846:	learn: 0.0347845	total: 7.5s	remaining: 237ms
4847:	learn: 0.0347523	total: 7.5s	remaining: 235ms
4848:	learn: 0.0347223	total: 7.5s	remaining: 234ms
4849:	learn: 0.0346874	total: 7.5s	remaining: 232ms
4850:	learn: 0.0346710	total: 7.5s	remaining: 230ms
4851:	learn: 0.0346377	total: 7.5s	remaining: 229ms
4852:	learn: 0.0345974	total: 7.5s	remaining: 227ms
4853:	learn: 0.0345967	total: 7.51s	remaining: 226ms
4854:	learn: 0.0345569	total: 7.51s	remaining: 224ms
4855:	learn: 0.0345078	total: 7.51s	remaining: 223ms
4856:	learn: 0.0344676	total: 7.51s	remaining: 221ms
4857:	learn: 0.0344462	total: 7.51s	remaining: 220ms
4858:	learn: 0.0344273	total: 7.51s	remaining: 218ms
4859:	learn: 0.0343938	total: 7.52s	remaining: 217ms


In [172]:
print("Alg --- MSE --- R2 --- Accuracy")

for s in [li_pred, gb_pred, cgb_pred, lgb_pred]:
    print([k for k in globals() if s is globals()[k]][0][:-5], round(mean_squared_error(np.round(s, 0), test_label.G3), 2), round(r2_score(test_label.G3, np.round(s, 0)), 2), round(accuracy_score(test_label.G3, np.round(s, 0)), 2), sep = "    ")

Alg --- MSE --- R2 --- Accuracy
li    6.96    0.21    0.14
gb    8.65    0.02    0.19
cgb    6.89    0.22    0.14
lgb    11.03    -0.25    0.12


In [173]:
blend_a=(li_pred+gb_pred+cgb_pred+lgb_pred)/4 #mean
blend_b=(li_pred*gb_pred*cgb_pred*lgb_pred)**(1/4) ##geom mean

blend_c=(gb_pred+cgb_pred+lgb_pred)/3 #only boost
blend_d=(gb_pred*cgb_pred*lgb_pred)**(1/3)

blend_e=(li_pred*acc(li_pred)+gb_pred*acc(gb_pred)+cgb_pred*acc(cgb_pred)+lgb_pred*acc(lgb_pred))/(acc(li_pred)+acc(gb_pred)+acc(cgb_pred)+acc(lgb_pred))
blend_f=(li_pred*r2(li_pred)+gb_pred*r2(gb_pred)+cgb_pred*r2(cgb_pred)+lgb_pred*r2(lgb_pred))/(r2(li_pred)+r2(gb_pred)+r2(cgb_pred)+r2(lgb_pred))
blend_g=(li_pred/mse(li_pred)+gb_pred/mse(gb_pred)+cgb_pred/mse(cgb_pred)+lgb_pred/mse(lgb_pred))/(1/mse(li_pred)+1/mse(gb_pred)+1/mse(cgb_pred)+1/mse(lgb_pred))

In [174]:
print("Alg ------ MSE --- R2 --- Accuracy")

for s in [globals()["blend_"+s] for s in ["a", "b", "c", "d", "e", "f", "g"]]:
    
    print([k for k in globals() if s is globals()[k]][1], round(mean_squared_error(np.round(s, 0), test_label.G3), 2), round(r2_score(test_label.G3, np.round(s, 0)), 2), round(accuracy_score(test_label.G3, np.round(s, 0)), 2), sep = "    ")

Alg ------ MSE --- R2 --- Accuracy
blend_a    6.7    0.24    0.17
blend_b    6.96    0.21    0.15
blend_c    7.93    0.1    0.16
blend_d    7.85    0.11    0.18
blend_e    6.63    0.25    0.17
blend_f    9.59    -0.09    0.21
blend_g    6.62    0.25    0.13


## Постройте решение на основе подхода Stacking

Правила:
- Реализуйте пайплайн обучения и предсказания (например, sklearn.pipeline или класс)
- Проведите оптимизацию пайплайна
- Оцените вклад каждого базового алгоритма в итоговое предсказание

In [148]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
svr_rbf = SVR(kernel='rbf')

stack_model = StackingCVRegressor(regressors=(li_reg, gb_reg, cgb_reg, lgb_reg), cv=kf, meta_regressor = RandomForestRegressor() ,
                            use_features_in_secondary=True,
                            store_train_meta_features=True,
                            shuffle=False,
                            random_state=42)

In [149]:
numerical_transformer = Pipeline(steps=[
       ('imputer', SimpleImputer(strategy='mean'))
       ,('scaler', StandardScaler())
       ,('RobustScaler', RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0), copy=True))  
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, X.columns)],
    remainder="passthrough"
)

In [150]:
stack_bundle = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', stack_model)
                     ])

stack_clf = TransformedTargetRegressor(regressor=stack_bundle, transformer=RobustScaler())

In [151]:
model = stack_clf.fit(X, y)

0:	learn: 0.7297930	total: 11.2ms	remaining: 55.8s
1:	learn: 0.7285221	total: 12.8ms	remaining: 32.1s
2:	learn: 0.7273436	total: 14.3ms	remaining: 23.8s
3:	learn: 0.7257844	total: 15.5ms	remaining: 19.4s
4:	learn: 0.7240507	total: 16.9ms	remaining: 16.9s
5:	learn: 0.7227359	total: 18.4ms	remaining: 15.3s
6:	learn: 0.7211112	total: 19.8ms	remaining: 14.1s
7:	learn: 0.7198327	total: 21.3ms	remaining: 13.3s
8:	learn: 0.7183591	total: 22.7ms	remaining: 12.6s
9:	learn: 0.7170347	total: 24.1ms	remaining: 12s
10:	learn: 0.7154418	total: 25.5ms	remaining: 11.6s
11:	learn: 0.7140164	total: 26.8ms	remaining: 11.1s
12:	learn: 0.7120232	total: 28.1ms	remaining: 10.8s
13:	learn: 0.7108817	total: 29.4ms	remaining: 10.5s
14:	learn: 0.7096392	total: 30.8ms	remaining: 10.2s
15:	learn: 0.7083330	total: 32.2ms	remaining: 10s
16:	learn: 0.7072101	total: 33.6ms	remaining: 9.84s
17:	learn: 0.7055777	total: 35ms	remaining: 9.69s
18:	learn: 0.7040766	total: 36.4ms	remaining: 9.55s
19:	learn: 0.7026178	total: 

222:	learn: 0.5100745	total: 320ms	remaining: 6.86s
223:	learn: 0.5095521	total: 322ms	remaining: 6.86s
224:	learn: 0.5087234	total: 323ms	remaining: 6.86s
225:	learn: 0.5082783	total: 325ms	remaining: 6.86s
226:	learn: 0.5076406	total: 327ms	remaining: 6.87s
227:	learn: 0.5068315	total: 328ms	remaining: 6.87s
228:	learn: 0.5059384	total: 330ms	remaining: 6.87s
229:	learn: 0.5057212	total: 330ms	remaining: 6.85s
230:	learn: 0.5049810	total: 332ms	remaining: 6.85s
231:	learn: 0.5041910	total: 333ms	remaining: 6.85s
232:	learn: 0.5035931	total: 335ms	remaining: 6.85s
233:	learn: 0.5026688	total: 337ms	remaining: 6.86s
234:	learn: 0.5020778	total: 338ms	remaining: 6.86s
235:	learn: 0.5013786	total: 340ms	remaining: 6.85s
236:	learn: 0.5005416	total: 341ms	remaining: 6.85s
237:	learn: 0.4994982	total: 342ms	remaining: 6.85s
238:	learn: 0.4987339	total: 344ms	remaining: 6.85s
239:	learn: 0.4980641	total: 345ms	remaining: 6.85s
240:	learn: 0.4974063	total: 347ms	remaining: 6.85s
241:	learn: 

430:	learn: 0.3963821	total: 632ms	remaining: 6.7s
431:	learn: 0.3959099	total: 634ms	remaining: 6.7s
432:	learn: 0.3953561	total: 635ms	remaining: 6.7s
433:	learn: 0.3950190	total: 637ms	remaining: 6.7s
434:	learn: 0.3945856	total: 639ms	remaining: 6.7s
435:	learn: 0.3938847	total: 640ms	remaining: 6.7s
436:	learn: 0.3935988	total: 642ms	remaining: 6.7s
437:	learn: 0.3931321	total: 644ms	remaining: 6.71s
438:	learn: 0.3927193	total: 645ms	remaining: 6.7s
439:	learn: 0.3922684	total: 647ms	remaining: 6.7s
440:	learn: 0.3918868	total: 648ms	remaining: 6.7s
441:	learn: 0.3913914	total: 650ms	remaining: 6.7s
442:	learn: 0.3909468	total: 651ms	remaining: 6.7s
443:	learn: 0.3900963	total: 653ms	remaining: 6.7s
444:	learn: 0.3894610	total: 655ms	remaining: 6.7s
445:	learn: 0.3889730	total: 657ms	remaining: 6.7s
446:	learn: 0.3885078	total: 658ms	remaining: 6.71s
447:	learn: 0.3878649	total: 660ms	remaining: 6.71s
448:	learn: 0.3875178	total: 661ms	remaining: 6.7s
449:	learn: 0.3870609	total:

627:	learn: 0.3153366	total: 940ms	remaining: 6.54s
628:	learn: 0.3150359	total: 942ms	remaining: 6.54s
629:	learn: 0.3147062	total: 943ms	remaining: 6.54s
630:	learn: 0.3145064	total: 945ms	remaining: 6.54s
631:	learn: 0.3140764	total: 946ms	remaining: 6.54s
632:	learn: 0.3137715	total: 948ms	remaining: 6.54s
633:	learn: 0.3133079	total: 950ms	remaining: 6.54s
634:	learn: 0.3128528	total: 951ms	remaining: 6.54s
635:	learn: 0.3123847	total: 953ms	remaining: 6.54s
636:	learn: 0.3119497	total: 954ms	remaining: 6.53s
637:	learn: 0.3115171	total: 956ms	remaining: 6.53s
638:	learn: 0.3111318	total: 957ms	remaining: 6.53s
639:	learn: 0.3107416	total: 959ms	remaining: 6.53s
640:	learn: 0.3104331	total: 960ms	remaining: 6.53s
641:	learn: 0.3097796	total: 962ms	remaining: 6.53s
642:	learn: 0.3093852	total: 963ms	remaining: 6.53s
643:	learn: 0.3090980	total: 964ms	remaining: 6.52s
644:	learn: 0.3088229	total: 966ms	remaining: 6.52s
645:	learn: 0.3085329	total: 967ms	remaining: 6.52s
646:	learn: 

837:	learn: 0.2459784	total: 1.25s	remaining: 6.22s
838:	learn: 0.2456398	total: 1.25s	remaining: 6.22s
839:	learn: 0.2453928	total: 1.25s	remaining: 6.21s
840:	learn: 0.2449256	total: 1.26s	remaining: 6.21s
841:	learn: 0.2446988	total: 1.26s	remaining: 6.21s
842:	learn: 0.2443861	total: 1.26s	remaining: 6.21s
843:	learn: 0.2440697	total: 1.26s	remaining: 6.21s
844:	learn: 0.2436801	total: 1.26s	remaining: 6.21s
845:	learn: 0.2434413	total: 1.26s	remaining: 6.21s
846:	learn: 0.2431886	total: 1.26s	remaining: 6.21s
847:	learn: 0.2430111	total: 1.27s	remaining: 6.2s
848:	learn: 0.2428055	total: 1.27s	remaining: 6.2s
849:	learn: 0.2424955	total: 1.27s	remaining: 6.2s
850:	learn: 0.2422571	total: 1.27s	remaining: 6.2s
851:	learn: 0.2420385	total: 1.27s	remaining: 6.2s
852:	learn: 0.2418475	total: 1.27s	remaining: 6.2s
853:	learn: 0.2415891	total: 1.28s	remaining: 6.2s
854:	learn: 0.2413360	total: 1.28s	remaining: 6.2s
855:	learn: 0.2409988	total: 1.28s	remaining: 6.19s
856:	learn: 0.240638

...

4178:	learn: 0.0267095	total: 6.76s	remaining: 1.33s
4179:	learn: 0.0266997	total: 6.76s	remaining: 1.33s
4180:	learn: 0.0266751	total: 6.76s	remaining: 1.32s
4181:	learn: 0.0266565	total: 6.76s	remaining: 1.32s
4182:	learn: 0.0266336	total: 6.76s	remaining: 1.32s
4183:	learn: 0.0266088	total: 6.77s	remaining: 1.32s
4184:	learn: 0.0265891	total: 6.77s	remaining: 1.32s
4185:	learn: 0.0265646	total: 6.77s	remaining: 1.32s
4186:	learn: 0.0265593	total: 6.77s	remaining: 1.31s
4187:	learn: 0.0265290	total: 6.77s	remaining: 1.31s
4188:	learn: 0.0265023	total: 6.78s	remaining: 1.31s
4189:	learn: 0.0264862	total: 6.78s	remaining: 1.31s
4190:	learn: 0.0264672	total: 6.78s	remaining: 1.31s
4191:	learn: 0.0264449	total: 6.78s	remaining: 1.31s
4192:	learn: 0.0264274	total: 6.78s	remaining: 1.3s
4193:	learn: 0.0264016	total: 6.78s	remaining: 1.3s
4194:	learn: 0.0263805	total: 6.78s	remaining: 1.3s
4195:	learn: 0.0263679	total: 6.79s	remaining: 1.3s
4196:	learn: 0.0263392	total: 6.79s	remaining: 1.3

4369:	learn: 0.0233637	total: 7.07s	remaining: 1.02s
4370:	learn: 0.0233467	total: 7.07s	remaining: 1.02s
4371:	learn: 0.0233327	total: 7.07s	remaining: 1.02s
4372:	learn: 0.0233066	total: 7.08s	remaining: 1.01s
4373:	learn: 0.0232970	total: 7.08s	remaining: 1.01s
4374:	learn: 0.0232788	total: 7.08s	remaining: 1.01s
4375:	learn: 0.0232694	total: 7.08s	remaining: 1.01s
4376:	learn: 0.0232526	total: 7.08s	remaining: 1.01s
4377:	learn: 0.0232301	total: 7.08s	remaining: 1.01s
4378:	learn: 0.0232261	total: 7.08s	remaining: 1s
4379:	learn: 0.0232137	total: 7.09s	remaining: 1s
4380:	learn: 0.0232074	total: 7.09s	remaining: 1s
4381:	learn: 0.0231939	total: 7.09s	remaining: 1s
4382:	learn: 0.0231755	total: 7.09s	remaining: 998ms
4383:	learn: 0.0231574	total: 7.09s	remaining: 997ms
4384:	learn: 0.0231448	total: 7.1s	remaining: 995ms
4385:	learn: 0.0231333	total: 7.1s	remaining: 994ms
4386:	learn: 0.0231209	total: 7.1s	remaining: 992ms
4387:	learn: 0.0231160	total: 7.1s	remaining: 990ms
4388:	lea

4560:	learn: 0.0204932	total: 7.38s	remaining: 711ms
4561:	learn: 0.0204755	total: 7.38s	remaining: 709ms
4562:	learn: 0.0204565	total: 7.39s	remaining: 707ms
4563:	learn: 0.0204477	total: 7.39s	remaining: 706ms
4564:	learn: 0.0204401	total: 7.39s	remaining: 704ms
4565:	learn: 0.0204346	total: 7.39s	remaining: 703ms
4566:	learn: 0.0204231	total: 7.39s	remaining: 701ms
4567:	learn: 0.0204105	total: 7.39s	remaining: 699ms
4568:	learn: 0.0204028	total: 7.4s	remaining: 698ms
4569:	learn: 0.0203826	total: 7.4s	remaining: 696ms
4570:	learn: 0.0203788	total: 7.4s	remaining: 695ms
4571:	learn: 0.0203559	total: 7.4s	remaining: 693ms
4572:	learn: 0.0203440	total: 7.4s	remaining: 691ms
4573:	learn: 0.0203334	total: 7.41s	remaining: 690ms
4574:	learn: 0.0203154	total: 7.41s	remaining: 688ms
4575:	learn: 0.0203079	total: 7.41s	remaining: 687ms
4576:	learn: 0.0202945	total: 7.41s	remaining: 685ms
4577:	learn: 0.0202825	total: 7.41s	remaining: 683ms
4578:	learn: 0.0202711	total: 7.41s	remaining: 682m

4742:	learn: 0.0179664	total: 7.7s	remaining: 417ms
4743:	learn: 0.0179593	total: 7.7s	remaining: 416ms
4744:	learn: 0.0179410	total: 7.7s	remaining: 414ms
4745:	learn: 0.0179237	total: 7.7s	remaining: 412ms
4746:	learn: 0.0179170	total: 7.71s	remaining: 411ms
4747:	learn: 0.0179064	total: 7.71s	remaining: 409ms
4748:	learn: 0.0178993	total: 7.71s	remaining: 407ms
4749:	learn: 0.0178880	total: 7.71s	remaining: 406ms
4750:	learn: 0.0178804	total: 7.71s	remaining: 404ms
4751:	learn: 0.0178739	total: 7.71s	remaining: 403ms
4752:	learn: 0.0178579	total: 7.71s	remaining: 401ms
4753:	learn: 0.0178385	total: 7.72s	remaining: 399ms
4754:	learn: 0.0178262	total: 7.72s	remaining: 398ms
4755:	learn: 0.0178122	total: 7.72s	remaining: 396ms
4756:	learn: 0.0177928	total: 7.72s	remaining: 394ms
4757:	learn: 0.0177856	total: 7.72s	remaining: 393ms
4758:	learn: 0.0177751	total: 7.72s	remaining: 391ms
4759:	learn: 0.0177627	total: 7.73s	remaining: 390ms
4760:	learn: 0.0177512	total: 7.74s	remaining: 389

4903:	learn: 0.0159580	total: 8.01s	remaining: 157ms
4904:	learn: 0.0159445	total: 8.02s	remaining: 155ms
4905:	learn: 0.0159354	total: 8.02s	remaining: 154ms
4906:	learn: 0.0159240	total: 8.02s	remaining: 152ms
4907:	learn: 0.0159099	total: 8.02s	remaining: 150ms
4908:	learn: 0.0159003	total: 8.02s	remaining: 149ms
4909:	learn: 0.0158948	total: 8.02s	remaining: 147ms
4910:	learn: 0.0158828	total: 8.03s	remaining: 145ms
4911:	learn: 0.0158698	total: 8.03s	remaining: 144ms
4912:	learn: 0.0158633	total: 8.03s	remaining: 142ms
4913:	learn: 0.0158486	total: 8.03s	remaining: 141ms
4914:	learn: 0.0158318	total: 8.03s	remaining: 139ms
4915:	learn: 0.0158136	total: 8.03s	remaining: 137ms
4916:	learn: 0.0158034	total: 8.04s	remaining: 136ms
4917:	learn: 0.0157980	total: 8.04s	remaining: 134ms
4918:	learn: 0.0157882	total: 8.04s	remaining: 132ms
4919:	learn: 0.0157755	total: 8.04s	remaining: 131ms
4920:	learn: 0.0157633	total: 8.04s	remaining: 129ms
4921:	learn: 0.0157510	total: 8.04s	remaining:

In [152]:
model_pred = model.predict(test_label.drop('G3',axis=1))

In [156]:
print("Alg ---- MSE --- R2 --- Accuracy")

for s in [model_pred]:
    
    print("stack", round(mean_squared_error(np.round(s, 0), test_label.G3), 2), round(r2_score(test_label.G3, np.round(s, 0)), 2), round(accuracy_score(test_label.G3, np.round(s, 0)), 2), sep = "    ")

Alg ---- MSE --- R2 --- Accuracy
stack    6.91    0.21    0.26


In [154]:
model.get_params()

{'check_inverse': True,
 'func': None,
 'inverse_func': None,
 'regressor__memory': None,
 'regressor__steps': [('preprocessor',
   ColumnTransformer(remainder='passthrough',
                     transformers=[('num',
                                    Pipeline(steps=[('imputer', SimpleImputer()),
                                                    ('scaler', StandardScaler()),
                                                    ('RobustScaler',
                                                     RobustScaler())]),
                                    Index(['sex', 'age', 'address', 'Medu', 'traveltime', 'studytime', 'famrel',
          'freetime', 'goout', 'Dalc', 'health', 'absences', 'school_GP',
          'famsize_GT3', 'Pstatus_A', 'Mjob_at_home', 'Mjob_health', 'Mjob_other',
          'Mjob_services', 'Mjob_teacher', 'Fjob_at_home', 'Fjob_health',
          'Fjob_other', 'Fjob_teacher', 'reason_course', 'reason_other',
          'reason_reputation', 'guardian_father', 'guardian_